# VisionEdge — NVDEC Hardware Decode Build & Test (Colab)

Builds ffmpeg from source with CUDA/NVDEC support, rebuilds PyAV against
it, and tests the full zero-copy pipeline (real hardware decode ->
TensorRT inference -> CUDA-kernel drawing) end to end.

**Honest expectations before you start:**
- This is a real compile-from-source process, not a quick pip install —
  budget 30-60+ minutes, most of it spent in the ffmpeg build step.
- Real risk of hitting build errors that need live troubleshooting —
  this is not guaranteed to succeed in one pass.
- If you're short on time before a demo/review: `test_zero_copy_gpu.py`
  (a separate, already-working notebook) already proves TensorRT +
  the CUDA kernel work correctly, without needing this. Treat this
  notebook as "if there's time," not a blocker.

Before running: **Runtime -> Change runtime type -> T4 GPU -> Save.**

## 1. Confirm GPU + CUDA version

In [1]:
!nvidia-smi
!nvcc --version

Sat Aug  8 07:48:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   65C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Upload and unzip the project code

Upload your code zip here. **Do not rely on a video bundled inside this
zip** — a real corruption issue was found doing that (`moov atom not
found` errors); videos get uploaded separately and directly in step 6.

In [2]:
import glob, os
from google.colab import files

uploaded = files.upload()
zips = sorted(glob.glob("/content/visionedge*.zip"), key=os.path.getmtime)
latest = zips[-1]
print("Using:", latest)

!unzip -oq "{latest}"
%cd /content/visionedge/backend
!ls

Saving visionedge.zip to visionedge.zip
Using: /content/visionedge.zip
/content/visionedge/backend
benchmark  __init__.py	  requirements-gpu.txt	  tests
core	   main.py	  requirements.txt	  test_zero_copy_gpu.py
decoder    orchestration  streaming
detector   pipeline	  test_nvdec_pipeline.py


## 3. Install base + GPU Python dependencies

We'll uninstall and rebuild `av` specifically in step 5 — installing it
normally here first is fine, it just gets replaced later.

In [3]:
!pip install -q aiohttp aiortc av opencv-python-headless onnx ultralytics
!pip install -q "tensorrt<11" pynvml
!pip install -q cupy-cuda12x
!python -c "import tensorrt as trt; print('TensorRT', trt.__version__)"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.7/93.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 35.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 13.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

## 4. Restore the custom FFmpeg build

In [4]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

archive = Path("/content/drive/MyDrive/ffmpeg_cuda_build.tar.gz")

if not archive.exists():
    raise FileNotFoundError(f"Archive not found: {archive}")

print(f"✓ Found: {archive}")
print(f"Size: {archive.stat().st_size / (1024**2):.2f} MB")

!rm -rf /content/ffmpeg_build
!tar -xzf /content/drive/MyDrive/ffmpeg_cuda_build.tar.gz -C /
print("✓ Restored /content/ffmpeg_build")

Mounted at /content/drive
✓ Found: /content/drive/MyDrive/ffmpeg_cuda_build.tar.gz
Size: 91.69 MB
✓ Restored /content/ffmpeg_build


## 5. Configure the environment

In [5]:
import os

os.environ["PATH"] = "/content/ffmpeg_build/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = "/content/ffmpeg_build/lib:" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["PKG_CONFIG_PATH"] = "/content/ffmpeg_build/lib/pkgconfig"

print("PATH configured")
print("LD_LIBRARY_PATH configured")
print("PKG_CONFIG_PATH =", os.environ["PKG_CONFIG_PATH"])

PATH configured
LD_LIBRARY_PATH configured
PKG_CONFIG_PATH = /content/ffmpeg_build/lib/pkgconfig


## 6. Rebuild PyAV from source against your custom ffmpeg

Not the pip binary wheel — that's the whole point of this notebook.

In [6]:
%%bash
export PATH=/content/ffmpeg_build/bin:$PATH
export LD_LIBRARY_PATH=/content/ffmpeg_build/lib:$LD_LIBRARY_PATH
export PKG_CONFIG_PATH=/content/ffmpeg_build/lib/pkgconfig

pip uninstall -y av || true
pip install av --no-binary av --no-cache-dir

Found existing installation: av 17.1.0
Uninstalling av-17.1.0:
  Successfully uninstalled av-17.1.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 157.0 MB/s eta 0:00:00
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for av: filename=av-18.0.0-cp311-abi3-linux_x86_64.whl size=8487346 sha256=56f3ca4810eda758cd2225dbd47ffe7e66c18b45b0a6230830ba19fdfc6f1dc9
  Stored in directory: /tmp/pip-ephem-wheel-cache-tjn9og_w/wheels/df/68/a2/b46ccc0c0eb8c938bbc83efc3f8c5b78d3d0cd005cbf9c23bb
Successfully built av


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
aiortc 1.15.0 requires av<18.0.0,>=14.0.0, but you have av 18.0.0 which is incompatible.


## 7. Verify NVDEC is actually active

The real test — don't skip this. Look for `cuda` entries with
`is_supported=True` in the output.

In [12]:
!python -m av --hwconfigs

Hardware configs:
    av1
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x783bbb014910>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x783bbb0148f0>
    av1_cuvid
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x783bbacf21e0>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x783bbacf21c0>
    h263
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x783bbb01dfd0>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x783bbb01dfb0>
    h263p
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x783bbb01dfd0>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x783bbb01dfb0>
    h264
        <av.HWConfig device_type=cuda format=cuda is_supported=True at 0x783bbb01e0d0>
        <av.HWConfig device_type=cuda format=cuarray is_supported=True at 0x783bbb01e0b0>
    h264_cuvid
        <av.HWConfig device_type

## 10. Get YOLO weights + build the TensorRT engine

Skip if you already have a built .engine file and just re-uploaded it into engines/.

In [13]:
import os
from ultralytics import YOLO

os.makedirs("../models", exist_ok=True)
model = YOLO("yolov10n.pt")
!mv yolov10n.pt ../models/yolov10n.pt

from google.colab import files

uploaded = files.upload()

!find . -name "__pycache__" -exec rm -rf {} +
!python -m detector.export_onnx --weights ../models/yolov10n.pt --output ../onnx/yolov10n.onnx
!python -m detector.build_engine --onnx ../onnx/yolov10n.onnx --engine ../engines/yolov10n_fp16.engine --precision fp16

Creating new Ultralytics Settings v0.0.7 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


Saving traffic_4k.mp4 to traffic_4k.mp4
/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
2026-08-08 08:03:15,193 [INFO] Loading PyTorch weights from ../models/yolov10n.pt
2026-08-08 08:03:15,265 [INFO] Exporting to ONNX (imgsz=(640, 640), opset=17)...
Ultralytics 8.4.116 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
YOLOv10n summary (fused): 101 layers, 2,299,264 parameters, 0 gradients, 6.8 GFLOPs

PyTorch: starting from '../models/yolov10n.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 300, 6) (5.6 MB)
requirements: Ultralytics requirements 

## 11. Ensure test_nvdec_pipeline.py is the current version

In [14]:
%%writefile test_nvdec_pipeline.py
"""
test_nvdec_pipeline.py

The full end-to-end test: HardwareFrameProvider (NVDEC decode) ->
Detector (TensorRT) -> cuda_draw (CUDA kernel), exactly as
zero_copy_pipeline.py orchestrates them. Unlike test_zero_copy_gpu.py,
this one does NOT bypass decode — it's only meaningful once PyAV has
been rebuilt from source against a CUDA-enabled ffmpeg (see
docs/SETUP.md and the NVDEC Colab notebook).

*** REQUIRES: NVIDIA GPU + a from-source PyAV build with NVDEC support.
Run test_zero_copy_gpu.py first if you haven't already — it isolates
TensorRT + the CUDA kernel from the decode question entirely, and is a
faster way to confirm those two are solid before adding decode into
the mix here. ***

Usage (from backend/, on a GPU machine with NVDEC-enabled PyAV):
    python test_nvdec_pipeline.py --video ../sample_media/traffic_4k.mp4 \\
        --engine ../engines/yolov10n_fp16.engine --num-frames 30
"""

import argparse
import logging

import cv2

from core.config import MODEL

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
log = logging.getLogger("test_nvdec_pipeline")


def main():
    parser = argparse.ArgumentParser(description="Test the full NVDEC zero-copy pipeline")
    parser.add_argument("--video", required=True)
    parser.add_argument("--engine", default=MODEL.engine_path)
    parser.add_argument("--num-frames", type=int, default=30)
    parser.add_argument("--save-sample", default="nvdec_pipeline_sample.jpg",
                         help="Path to save one annotated frame as proof, or '' to skip")
    args = parser.parse_args()

    from pipeline.zero_copy_pipeline import ZeroCopyPipeline

    log.info("Attempting to open %s with hardware (NVDEC) decode...", args.video)
    try:
        pipeline = ZeroCopyPipeline(source=args.video, engine_path=args.engine)
    except RuntimeError as e:
        log.error("NVDEC decode is not active: %s", e)
        log.error("This is expected unless PyAV has been rebuilt from source against a "
                   "CUDA-enabled ffmpeg. Run test_zero_copy_gpu.py instead to test "
                   "TensorRT + the CUDA kernel without needing real hardware decode.")
        raise SystemExit(1)

    total_detections = 0
    last_annotated_gpu = None

    def on_frame(annotated_gpu, detections, frame_index):
        nonlocal total_detections, last_annotated_gpu
        total_detections += len(detections)
        last_annotated_gpu = annotated_gpu
        if (frame_index + 1) % 10 == 0:
            log.info("frame=%d  detections_this_frame=%d", frame_index + 1, len(detections))

    pipeline.run(on_frame=on_frame, max_frames=args.num_frames)
    pipeline.close()

    log.info("=" * 60)
    log.info("Frames processed: %d", pipeline.stats.frames_processed)
    log.info("Total detections across all frames: %d", total_detections)
    log.info("Avg FPS: %.1f", pipeline.stats.avg_fps)
    log.info("  decode+preprocess: %.1fms/frame  inference: %.1fms/frame  draw: %.1fms/frame",
              1000 * pipeline.stats.total_decode_s / max(pipeline.stats.frames_processed, 1),
              1000 * pipeline.stats.total_inference_s / max(pipeline.stats.frames_processed, 1),
              1000 * pipeline.stats.total_draw_s / max(pipeline.stats.frames_processed, 1))
    log.info("=" * 60)
    log.info("Result: PASS — full NVDEC zero-copy pipeline runs end-to-end on real GPU hardware.")

    if args.save_sample and last_annotated_gpu is not None:
        import cupy as cp
        annotated_host = cp.asnumpy(last_annotated_gpu)
        cv2.imwrite(args.save_sample, cv2.cvtColor(annotated_host, cv2.COLOR_RGB2BGR))
        log.info("Saved sample annotated frame to %s", args.save_sample)


if __name__ == "__main__":
    main()


Overwriting test_nvdec_pipeline.py


In [16]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 53.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.2/99.2 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 10.2 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2026.1-cp312-cp312-linux_x86_64.whl size=659497 sha256=5b27d06b9dfee9f3b19f12a6012631644887e4d61cb4c3f08c376fab4a85c577
  Stored in directory: /root/.cache/pip/wheels/90/2a/71/75ec0cc316cc0ff494bfffa2935e02580129cb7f859a0cfd8f
Successfully built pycuda


## 12. Run the full NVDEC zero-copy pipeline test

If steps 1-7 succeeded, this exercises real hardware decode -> TensorRT
-> CUDA drawing, end to end, on real GPU hardware.

In [17]:
!python test_nvdec_pipeline.py --video traffic_4k.mp4 --engine ../engines/yolov10n_fp16.engine --num-frames 30

2026-08-08 08:14:18,733 [INFO] Attempting to open traffic_4k.mp4 with hardware (NVDEC) decode...
2026-08-08 08:14:19,589 [INFO] Loading TensorRT engine from ../engines/yolov10n_fp16.engine
2026-08-08 08:14:19,732 [INFO] Detector ready. Input size=(640, 640)
2026-08-08 08:14:22,285 [INFO] frame=10  detections_this_frame=7
2026-08-08 08:14:22,492 [INFO] frame=20  detections_this_frame=6
2026-08-08 08:14:22,711 [INFO] frame=30  detections_this_frame=7
2026-08-08 08:14:22,725 [INFO] ============================================================
2026-08-08 08:14:22,725 [INFO] Frames processed: 30
2026-08-08 08:14:22,726 [INFO] Total detections across all frames: 1664
2026-08-08 08:14:22,726 [INFO] Avg FPS: 16.3
2026-08-08 08:14:22,726 [INFO]   decode+preprocess: 50.8ms/frame  inference: 6.8ms/frame  draw: 3.8ms/frame
2026-08-08 08:14:22,726 [INFO] ============================================================
2026-08-08 08:14:22,726 [INFO] Result: PASS — full NVDEC zero-copy pipeline runs end-t

## 13. Download the annotated sample frame

In [18]:
from google.colab import files
files.download("nvdec_pipeline_sample.jpg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## If step 12 raises a RuntimeError about NVDEC not being active

That means steps 1-7 didn't fully succeed — PyAV is still running without
CUDA support. Two options:
- Re-check step 7's output carefully for `is_supported=True` under a
  `cuda` entry before assuming the build worked.
- Fall back to `test_zero_copy_gpu.py` (separate notebook) — it already
  proves TensorRT + the CUDA kernel work correctly without needing NVDEC,
  which is credible, demo-ready evidence on its own.